# Statistical Analysis

### Common Preprocessing

In [1]:
# Import necessary libraries

import pandas as pd
import numpy as np

In [2]:
# Load the data files

life_total = pd.read_csv(
    "../data/total_life_expectancy_at_birth.csv", skiprows=4
)
life_male = pd.read_csv(
    "../data/male_life_expectancy_at_birth.csv", skiprows=4
)
life_female = pd.read_csv(
    "../data/female_life_expectancy_at_birth.csv", skiprows=4
)
fertility = pd.read_csv("../data/fertility_rate_total.csv", skiprows=4)

# Load the CSV with country metadata
metadata = pd.read_csv("../data/country_metadata.csv")

In [3]:
metadata

,Country Code,Region,IncomeGroup,SpecialNotes,TableName,Unnamed: 5
0,ABW,Latin America & Caribbean,High income,NaN,Aruba,NaN
1,AFE,NaN,NaN,"26 countries, stretching from the Red Sea in t...",Africa Eastern and Southern,NaN
2,AFG,Middle East & North Africa,Low income,The reporting period for national accounts dat...,Afghanistan,NaN
3,AFW,NaN,NaN,"22 countries, stretching from the westernmost ...",Africa Western and Central,NaN
4,AGO,Sub-Saharan Africa,Lower middle income,The World Bank systematically assesses the app...,Angola,NaN
...,...,...,...,...,...,...
259,XKX,Europe & Central Asia,Upper middle income,NaN,Kosovo,NaN
260,YEM,Middle East & North Africa,Low income,The World Bank systematically assesses the app...,"Yemen, Rep.",NaN
261,ZAF,Sub-Saharan Africa,Upper middle income,Fiscal year end: March 31; reporting period fo...,South Africa,NaN
262,ZMB,Sub-Saharan Africa,Lower middle income,National accounts data were rebased to reflect...,Zambia,NaN


I found that the metadata file contains blank values in the `Region` and `IncomeGroup` columns for aggregate rows such as 'World', 'Arab World', 'EUU' (European Union Aggregate) etc. These aggregates distort country‑level analyses, so to restrict the dataset to individual countries, I filter out any rows with a missing `IncomeGroup`.

In [4]:
metadata = metadata[
    metadata["IncomeGroup"].notna()
    & (metadata["IncomeGroup"].str.strip() != "")
]

In [5]:
valid_codes = set(metadata["Country Code"])
len(valid_codes)

217

After filtering out aggregate entries, the dataset contains 217 individual countries for further analysis. 
To ensure the analysis only covers valid country entries, I filter each dataset (`life_total`, `life_male`, `life_female`, `fertility`) by keeping only rows whose `Country Code` is in the list of valid codes. I also drop blank columns (`2025` and `Unnamed: 70`) that are not part of the analysis. Finally, I define the list of years from 1960 to 2023 to use consistently across all visualizations.


In [6]:
for df_name in [life_total, life_male, life_female, fertility]:
    df_name.drop(
        df_name[~df_name["Country Code"].isin(valid_codes)].index, inplace=True
    )
    df_name.drop(columns=["2025", "Unnamed: 70"], inplace=True)

years = [str(y) for y in range(1960, 2024)]

### Task 1

**For which income group has the difference in average life expectancy of men and women changed the most between 1960 and 2023?**

In the following code
1. I merge the `IncomeGroup` information from the metadata into both the male and female life expectancy datasets.
2. Then, for each income group, I select only those countries that have valid data for both 1960 and 2023 and are present in both male and female datasets. This ensures that comparisons are consistent across sexes and years. 
3. For each group, I compute the average life expectancy for males and females in 1960 and 2023, calculate the female–male gap in each year, and measure how that gap has changed over time. 
4. Finally, I collect these results into a DataFrame and sort them by the largest absolute change in the gap, highlighting where disparities have shifted the most.


In [7]:
# Merge income group information

male = life_male.merge(
    metadata[["Country Code", "IncomeGroup"]], on="Country Code", how="inner"
)

female = life_female.merge(
    metadata[["Country Code", "IncomeGroup"]], on="Country Code", how="inner"
)

results = []

# Iterate over each income group

for group in metadata["IncomeGroup"].dropna().unique():

    # Select countries belonging to this particular income group

    male_grp = male[male["IncomeGroup"] == group].copy()
    female_grp = female[female["IncomeGroup"] == group].copy()

    # Keep only countries with data for BOTH years and BOTH sexes

    male_grp = male_grp.dropna(subset=["1960", "2023"])

    female_grp = female_grp.dropna(subset=["1960", "2023"])

    # Find countries present in BOTH male and female datasets

    common_countries = set(male_grp["Country Code"]) & set(
        female_grp["Country Code"]
    )

    male_grp = male_grp[male_grp["Country Code"].isin(common_countries)]

    female_grp = female_grp[female_grp["Country Code"].isin(common_countries)]

    # Calculate average life expectancy for each sex

    male_1960 = male_grp["1960"].mean()
    female_1960 = female_grp["1960"].mean()

    male_2023 = male_grp["2023"].mean()
    female_2023 = female_grp["2023"].mean()

    # Calculate female-male gap

    gap_1960 = female_1960 - male_1960
    gap_2023 = female_2023 - male_2023

    # Calculate change in the gap

    change = gap_2023 - gap_1960

    # Store results

    results.append(
        {
            "IncomeGroup": group,
            "Gap1960": gap_1960,
            "Gap2023": gap_2023,
            "GapChange": change,
            "AbsChange": abs(change),
        }
    )

result_df = pd.DataFrame(results)

# Sort by largest absolute change
result_df = result_df.sort_values("AbsChange", ascending=False).reset_index(
    drop=True
)

result_df

,IncomeGroup,Gap1960,Gap2023,GapChange,AbsChange
0,Upper middle income,4.262407,5.928627,1.666220,1.666220
1,Low income,2.839840,4.399200,1.559360,1.559360
2,Lower middle income,3.196609,4.586022,1.389413,1.389413
3,High income,5.239274,5.192611,-0.046663,0.046663


As shown in the resulting dataframe, the **Upper middle income** group exhibits the largest average change in the female–male life expectancy gap between 1960 and 2023.


### Task 2

**For which income group has variability in life expectancy at birth changed the most between 1960 and 2023?**

The goal here is to identify which income group has experienced the greatest change in variability of life expectancy at birth between 1960 and 2023. To measure variability, I use the **standard deviation (sd)** of life expectancy values within each group. Standard deviation is appropriate because it quantifies how spread out the values are around the mean, allowing us to see whether countries within the same income group have become more similar or more divergent over time.

I calculate the standard deviation for each group in 1960 and again in 2023, then compare the difference. A larger change indicates that the distribution of life expectancy within that group has shifted more significantly.

**Note on `ddof=0`:**  
By default, `pandas` uses `ddof=1` to compute the sample standard deviation. Here I set `ddof=0` to compute the **population standard deviation**, since the dataset includes all countries in each income group rather than a sample. This ensures the variability measure reflects the entire population of countries in each group.


In [8]:
# Merge income group information
life = life_total.merge(
    metadata[["Country Code", "IncomeGroup"]], on="Country Code", how="inner"
)

results = []

for group in metadata["IncomeGroup"].dropna().unique():

    # Select the income group

    grp = life[life["IncomeGroup"] == group].copy()

    # Keep only countries with data for BOTH 1960 and 2023

    grp = grp.dropna(subset=["1960", "2023"])

    # Calculate standard deviation for each year

    sd_1960 = grp["1960"].std(ddof=0)
    sd_2023 = grp["2023"].std(ddof=0)

    # Calculate change in variability

    change = sd_2023 - sd_1960

    # Store results

    results.append(
        {
            "IncomeGroup": group,
            "CountryCount": grp["Country Code"].nunique(),
            "SD1960": sd_1960,
            "SD2023": sd_2023,
            "SDChange": change,
            "AbsChange": abs(change),
        }
    )

variability_df = pd.DataFrame(results)

# Sort by largest absolute change
variability_df = variability_df.sort_values(
    "AbsChange", ascending=False
).reset_index(drop=True)

variability_df

,IncomeGroup,CountryCount,SD1960,SD2023,SDChange,AbsChange
0,Upper middle income,59,7.485098,3.746142,-3.738955,3.738955
1,High income,85,6.869040,4.160662,-2.708378,2.708378
2,Low income,25,6.624919,4.507766,-2.117154,2.117154
3,Lower middle income,46,6.267166,4.879202,-1.387964,1.387964


As observed in the resulting dataframe, the **Upper middle income** group shows the greatest change in the standard deviation of life expectancy at birth between 1960 and 2023.

**Note:**  

When applying the filter that requires countries to have data available for both 1960 and 2023, certain entries are excluded from the analysis. Specifically, **Israel (ISR)** and **West Bank and Gaza (PSE)** got removed because their records do not contain values for the year 1960. 

The table below displays the year‑wise data for countries that were dropped under this filter in the total life expectancy dataset. The same pattern is observed in the male and female life expectancy datasets as well.


In [9]:
life[life["1960"].isna() | life["2023"].isna()][
    ["Country Name", "Country Code"] + years
]

,Country Name,Country Code,1960,1961,1962,1963,1964,1965,1966,1967,...,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
94,Israel,ISR,NaN,72.006585,72.112195,NaN,NaN,NaN,72.28561,71.509756,...,82.153659,82.05122,82.407317,82.55122,82.802439,82.804878,82.64878,82.50,82.700,83.195122
161,West Bank and Gaza,PSE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,72.781000,74.57700,74.917000,75.21600,75.380000,75.811000,74.99800,73.89,76.662,65.170000


### Task 3

**Which countries have the highest correlation between fertility rate and life expectancy at birth over the years (either positive or negative)? Which countries have the lowest correlation?**

The objective of this task is to determine which countries show the strongest relationship between fertility rate and life expectancy at birth over time. By computing the correlation coefficient for each country across the years 1960–2023, we can identify whether higher fertility is consistently associated with lower life expectancy (negative correlation), or whether any countries exhibit a positive relationship. Countries with the **highest absolute correlation** values represent the strongest link - either positive or negative, between these two measures, while those with the lowest values show little to no association.

**Note on filtering:**  
To ensure meaningful results, countries with fewer than 10 valid data points across the years are excluded. This avoids spurious correlations that could arise from very limited data.


In [10]:
# Merge fertility and life expectancy with metadata

life = life_total.merge(
    metadata[["Country Code"]], on="Country Code", how="inner"
)
fert = fertility.merge(
    metadata[["Country Code"]], on="Country Code", how="inner"
)

corr_results = []

# Iterate over each country

for code in life["Country Code"]:

    # Extract life expectancy and fertility rows for the country

    life_row = life[life["Country Code"] == code].iloc[0]
    fert_row = fert[fert["Country Code"] == code].iloc[0]

    # Convert values to numeric for all years

    x = pd.to_numeric(fert_row[years], errors="coerce")
    y = pd.to_numeric(life_row[years], errors="coerce")

    # Combine into a temporary DataFrame and drop missing values

    temp = pd.DataFrame({"fertility": x, "life": y}).dropna()

    # Skip countries with insufficient data (<10 overlapping years)

    if len(temp) < 10:
        print(f"Not much data for {code}, skipping...")
        continue

    # Calculate correlation

    corr = temp["fertility"].corr(temp["life"])

    # Store results

    corr_results.append(
        {
            "Country": life_row["Country Name"],
            "CountryCode": code,
            "Correlation": corr,
        }
    )

# Create DataFrame of correlations

corr_df = pd.DataFrame(corr_results)

In [11]:
# Strongest absolute corrrelations
corr_df["AbsCorrelation"] = corr_df["Correlation"].abs()
strongest_overall = (
    corr_df.sort_values("AbsCorrelation", ascending=False)
    .head(10)
    .reset_index(drop=True)
)


print("\n\nStrongest absolute correlation\n===========================")
print(strongest_overall)

# Weakest correlations (closest to zero)


weakest = (
    corr_df.sort_values("AbsCorrelation", ascending=True)
    .head(10)
    .reset_index(drop=True)
)


print("Weakest correlation:\n===========================")
print(weakest)



Strongest absolute correlation
                    Country CountryCode  Correlation  AbsCorrelation
0                     India         IND    -0.997255        0.997255
1         Brunei Darussalam         BRN    -0.994299        0.994299
2                     Ghana         GHA    -0.993698        0.993698
3                   Turkiye         TUR    -0.991585        0.991585
4  Turks and Caicos Islands         TCA    -0.990753        0.990753
5        Dominican Republic         DOM    -0.990069        0.990069
6                  Malaysia         MYS    -0.987550        0.987550
7          Egypt, Arab Rep.         EGY    -0.986969        0.986969
8                   Albania         ALB    -0.986553        0.986553
9                   Ecuador         ECU    -0.985878        0.985878
Weakest correlation:
                    Country CountryCode  Correlation  AbsCorrelation
0                   Belarus         BLR     0.020421        0.020421
1                  Eswatini         SWZ     0.039

From the correlation analysis, the countries with the **strongest correlations** (in descending order of magnitude, whether positive or negative) between fertility rate and life expectancy at birth are:

1. **India** (-0.998)  
2. Brunei Darussalam  
3. Ghana  
4. Turkiye  
5. Turks and Caicos Islands  

On the other hand, the countries with the **weakest correlations** (in ascending order) are:

1. **Belarus** (0.020)  
2. Eswatini  
3. Central African Republic  
4. Congo, Dem. Rep.  
5. Ukraine  

